# 04 — Exploration et comparaison de modèles

**Projet** : Prédiction d'attrition client (churn télécom) avec scikit-learn
**Modèle configuré** : `random_forest` (Forêt aléatoire scikit-learn)
**Pourquoi ce choix** : La forêt aléatoire est le meilleur compromis pédagogie/performance pour un premier modèle tabulaire : elle capture les non-linéarités et les interactions (contrat mensuel x jeune abonné x satisfaction basse) sans réglage fin, fournit une importance de features native et expose des probabilités exploitables pour le ciblage CRM. Elle sert aussi de référence pour comparer les autres stacks du même cas d'usage.

Ce notebook répond à la question que tout relecteur pose : *« pourquoi cet algorithme ? »*.
La réponse doit être **chiffrée** : baseline, comparaison d'algorithmes, sensibilité aux
hyperparamètres, courbe d'apprentissage et importance des features.

## Objectifs pédagogiques

1. Commencer par une **baseline** : sans elle, aucune performance n'est interprétable.
1. Comparer les algorithmes disponibles dans la stack via `available_algorithms()`.
1. Distinguer sous-apprentissage et sur-apprentissage avec une courbe d'apprentissage.
1. Lire l'importance des features pour décider quoi instrumenter ensuite.

**Objectifs transverses du dépôt**

- Composer un pipeline scikit-learn propre : ColumnTransformer, transformers custom, fit sur le train uniquement.
- Utiliser une classe abstraite BaseModel pour rendre le framework interchangeable.
- Lire des métriques de classification en contexte déséquilibré (ROC AUC, PR AUC, rappel, précision).

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.7)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

## 1. La baseline d'abord

In [ ]:
from src.evaluation.evaluator import Evaluator
from src.models import build_model
from src.training.losses_metrics import MetricCalculator, MetricInputs

BASE_MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"], algorithm="random_forest")
_ = BASE_MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    callbacks=[],
)

EVALUATOR = Evaluator.from_config(BASE_MODEL, CONFIG.model_dump(), NB_PATHS)
baseline_metrics = EVALUATOR.compare_to_baseline(
    PREPARED["X_val"], PREPARED["y_val"], baseline="dummy"
)

calculator = MetricCalculator(task=CONFIG.metrics.task, metrics=CONFIG.metrics.all_metrics)
model_metrics = calculator.evaluate(
    MetricInputs(
        y_true=PREPARED["y_val"],
        y_pred=BASE_MODEL.predict(PREPARED["X_val"]),
        y_proba=BASE_MODEL.predict_proba(PREPARED["X_val"]) if BASE_MODEL.supports_proba else None,
        X=PREPARED["X_val"],
    )
)

comparison = pd.DataFrame(
    {
        "modèle": ["random_forest", "baseline (classe majoritaire)"],
        CONFIG.metrics.primary: [
            model_metrics.get(CONFIG.metrics.primary, float("nan")),
            baseline_metrics.get(f"baseline_{CONFIG.metrics.primary}", float("nan")),
        ],
    }
)
comparison.round(4)

**Ce qu'il faut retenir**

- Un modèle qui ne bat pas la baseline n'apporte **aucune** valeur : il ne doit pas aller en production.
- La métrique primaire du projet est `roc_auc` (sens : maximize) ; le seuil de qualité déclaré est 0.7.
- La baseline est recalculée sur le **même** split de validation : la comparaison est loyale.

## 2. Comparaison des algorithmes de la stack (réglages par défaut)

In [ ]:
from src.models.factory import available_algorithms

ALGORITHMS = available_algorithms(CONFIG.metrics.task)
print(f"{len(ALGORITHMS)} algorithmes disponibles pour la tâche '{CONFIG.metrics.task}' :")
print(ALGORITHMS)

rows = []
# Comparaison **loyale** : chaque algorithme est construit avec ses réglages par défaut.
# Les `model.params` configurés sont réglés pour l'algorithme retenu ; les réutiliser
# ailleurs fausserait le classement (`max_leaf_nodes` bride une forêt aléatoire).
# Le modèle configuré et réglé est, lui, évalué en section 1.
for algorithm in ALGORITHMS:
    try:
        candidate = build_model(
            CONFIG, feature_names=PREPARED["feature_names"], algorithm=algorithm, params={}
        )
        result = candidate.fit(
            PREPARED["X_train"],
            PREPARED["y_train"],
            X_val=PREPARED["X_val"],
            y_val=PREPARED["y_val"],
            callbacks=[],
        )
        values = calculator.evaluate(
            MetricInputs(
                y_true=PREPARED["y_val"],
                y_pred=candidate.predict(PREPARED["X_val"]),
                y_proba=candidate.predict_proba(PREPARED["X_val"])
                if candidate.supports_proba
                else None,
                X=PREPARED["X_val"],
            )
        )
        rows.append(
            {
                "algorithme": algorithm,
                CONFIG.metrics.primary: values.get(CONFIG.metrics.primary, float("nan")),
                **{name: values.get(name, float("nan")) for name in CONFIG.metrics.secondary[:3]},
                "secondes": round(result.duration_seconds, 2),
            }
        )
    except Exception as error:  # un algorithme incompatible ne doit pas casser l'exploration
        rows.append(
            {
                "algorithme": algorithm,
                CONFIG.metrics.primary: float("nan"),
                "secondes": float("nan"),
            }
        )
        print(f"  ! {algorithm} ignoré : {type(error).__name__}: {error}")

# Le meilleur en tête, quel que soit le sens de la métrique (AUC : décroissant, RMSE : croissant).
meilleur_d_abord = str(CONFIG.metrics.direction) == "minimize"
ranking = (
    pd.DataFrame(rows)
    .sort_values(CONFIG.metrics.primary, ascending=meilleur_d_abord)
    .reset_index(drop=True)
)
ranking.round(4)

**Ce qu'il faut retenir**

- Le classement se lit **avec** le temps d'entraînement : un gain de 0.005 pour 20x plus lent est rarement rentable.
- Chaque algorithme est comparé **à réglages par défaut** : un algorithme perdant ici peut gagner une fois réglé (section 3).
- Un écart faible entre algorithmes indique que la limite vient des **données**, pas du modèle.
- Les valeurs manquantes (NaN) signalent une métrique non définie pour l'algorithme (ex. probabilités absentes).

In [ ]:
fig, axis = plt.subplots(figsize=(8.2, 0.45 * len(ranking) + 1.8))
axis.barh(ranking["algorithme"][::-1], ranking[CONFIG.metrics.primary][::-1], color="#005f73")
baseline_value = baseline_metrics.get(f"baseline_{CONFIG.metrics.primary}", float("nan"))
if np.isfinite(baseline_value):
    axis.axvline(baseline_value, color="#d1495b", linestyle="--", linewidth=1.2, label="baseline")
    axis.legend(fontsize=8)
axis.set_xlabel(f"{CONFIG.metrics.primary} (validation)")
axis.set_title("Comparaison des algorithmes")
fig.tight_layout()
plt.show()

## 3. Sensibilité aux hyperparamètres

In [ ]:
import itertools

GRID = {"n_estimators": [120, 300], "max_depth": [6, 14], "min_samples_leaf": [4, 16]}
combinations = list(itertools.product(*[GRID[name] for name in GRID]))
print(f"{len(combinations)} combinaisons testées sur {list(GRID)}")

rows = []
for combination in combinations:
    params = dict(zip(GRID, combination, strict=True))
    candidate = build_model(CONFIG, feature_names=PREPARED["feature_names"], params=params)
    _ = candidate.fit(
        PREPARED["X_train"],
        PREPARED["y_train"],
        X_val=PREPARED["X_val"],
        y_val=PREPARED["y_val"],
        callbacks=[],
    )
    values = calculator.evaluate(
        MetricInputs(
            y_true=PREPARED["y_val"],
            y_pred=candidate.predict(PREPARED["X_val"]),
            y_proba=candidate.predict_proba(PREPARED["X_val"])
            if candidate.supports_proba
            else None,
        )
    )
    row = {str(name): str(value) for name, value in params.items()}
    row[CONFIG.metrics.primary] = values.get(CONFIG.metrics.primary, float("nan"))
    rows.append(row)

grid_frame = pd.DataFrame(rows).sort_values(
    CONFIG.metrics.primary, ascending=str(CONFIG.metrics.direction) == "minimize"
)
grid_frame.round(4)

**Ce qu'il faut retenir**

- La meilleure ligne du tableau est un **candidat**, pas une décision : elle est choisie sur la validation.
- Un modèle très complexe qui n'améliore pas la validation sur-apprend : revenir au plus simple.
- Ces valeurs appartiennent dans `conf/model/default.yaml` (`model.params`), jamais dans le code.

## 4. Courbe d'apprentissage — faut-il plus de données ?

In [ ]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
curve = []
for fraction in fractions:
    size = max(int(len(PREPARED["X_train"]) * fraction), 30)
    subsample = PREPARED["X_train"].iloc[:size]
    labels = None if PREPARED["y_train"] is None else PREPARED["y_train"].iloc[:size]
    candidate = build_model(CONFIG, feature_names=PREPARED["feature_names"])
    train_result = candidate.fit(subsample, labels, callbacks=[])
    train_score = float(train_result.metrics.get(CONFIG.metrics.primary, float("nan")))
    val_score = float("nan")
    if PREPARED["X_val"] is not None and PREPARED["y_val"] is not None:
        val_values = calculator.evaluate(
            MetricInputs(
                y_true=PREPARED["y_val"],
                y_pred=candidate.predict(PREPARED["X_val"]),
                y_proba=candidate.predict_proba(PREPARED["X_val"])
                if candidate.supports_proba
                else None,
            )
        )
        val_score = float(val_values.get(CONFIG.metrics.primary, float("nan")))
    curve.append({"lignes": size, "train": train_score, "validation": val_score})

curve_frame = pd.DataFrame(curve)
fig, axis = plt.subplots(figsize=(6.6, 3.6))
axis.plot(curve_frame["lignes"], curve_frame["train"], marker="o", label="train")
axis.plot(curve_frame["lignes"], curve_frame["validation"], marker="s", label="validation")
axis.set_xlabel("lignes d'entraînement")
axis.set_ylabel(CONFIG.metrics.primary)
axis.set_title("Courbe d'apprentissage")
axis.legend(fontsize=8)
fig.tight_layout()
plt.show()
curve_frame.round(4)

**Ce qu'il faut retenir**

- Courbes **écartées et plates** ⇒ sur-apprentissage : régulariser plutôt qu'ajouter des données.
- Courbes **encore montantes** ⇒ plus de données aiderait vraiment (argument budgétaire chiffré).
- Un train parfait (1.0) avec une validation médiocre est le signal d'une fuite ou d'un modèle trop complexe.

## 5. Importance des features

In [ ]:
importance = EVALUATOR.feature_importance(PREPARED["X_val"], y=PREPARED["y_val"])
if importance.empty:
    print("Ce modèle n'expose pas d'importance (permutation indisponible sur ce split).")
else:
    display(importance.head(12).round(4))
    top = importance.head(12).iloc[::-1]
    fig, axis = plt.subplots(figsize=(7.6, 0.4 * len(top) + 1.4))
    axis.barh(top["feature"], top["importance"], color="#0a9396")
    axis.set_xlabel(f"importance ({top['method'].iloc[0] if 'method' in top else 'naine'})")
    axis.set_title("Top 12 des features")
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- L'importance **native** (Gini, coefficients) est rapide mais biaisée vers les variables à forte cardinalité.
- L'importance par **permutation** est plus lente mais model-agnostic : c'est celle à citer en comité.
- Une feature dominante impose une vigilance particulière sur sa disponibilité et sa dérive en production.

## Synthèse — choix du modèle

- Algorithme retenu par la configuration : **`random_forest`** (Forêt aléatoire scikit-learn).
- Justification documentée : La forêt aléatoire est le meilleur compromis pédagogie/performance pour un premier modèle tabulaire : elle capture les non-linéarités et les interactions (contrat mensuel x jeune abonné x satisfaction basse) sans réglage fin, fournit une importance de features native et expose des probabilités exploitables pour le ciblage CRM. Elle sert aussi de référence pour comparer les autres stacks du même cas d'usage.
- Alternatives évaluées ici : `logistic_regression : baseline interprétable, sensible au scaling et aux interactions`, `hist_gradient_boosting : généralement supérieur, plus coûteux à régler`, `gradient_boosting : variante historique, plus lente`, `extra_trees : plus de variance aléatoire, parfois plus robuste au bruit`, `svm (RBF) : bon sur petits volumes, coûteux au-delà de 100k lignes`.

**Règle de décision** : on retient le modèle le plus **simple** dont la métrique primaire est à
moins de ~1 point du meilleur, et dont le coût d'inférence est compatible avec `Score recalculé quotidiennement (batch nocturne) et exposé au CRM.`.

**Suite** : `05_training.ipynb` entraîne le modèle retenu dans les conditions de production.